# 03_silver_transform — Incremental Harmonize, Deduplicate, Validate

Streams **new** rows from `bronze/articles_raw` (checkpointed — 
`03_silver_transform` never re-reads a bronze row it's already processed) 
and, for each new micro-batch: parses it into the shared article schema, 
applies CHK-6 duplicate handling (dedupe on canonical URL within the 
batch, log merges), and schema validation.

**Write mode:** validated rows are `MERGE`d into `silver/articles`, keyed 
on `canonical_url` — a real-world article that reappears in a later batch 
(its source re-published it, a summary got edited) refreshes its existing 
row instead of duplicating it; a genuinely new article is inserted. 
Rejected rows are appended to `silver/articles_rejected` as an audit log.

**Trade-off worth knowing about:** CHK-6's canonical-URL dedup only runs 
*within* each new micro-batch, not across the full table. Two 
near-duplicate rows that land in the *same* batch still get merged; two 
near-duplicates that land in *separate* runs won't be caught by CHK-6 — 
but the `canonical_url` merge key means a genuinely repeated article (same 
URL) still collapses into one row across runs either way.

---
## Business rules added in this revision

| Rule | What changed |
|---|---|
| **CHK-2** | A `http://` URL is auto-upgraded to `https://` before validation ("fix automatically", per the sheet) instead of being rejected outright. Only URLs that are still missing/malformed after that fail. |
| **CHK-3 + TR-8** | These two source rules contradict each other (CHK-3: reject null `Published_date`; TR-8: impute null with `today()`). Resolved by applying TR-8's imputation first, so CHK-3's real enforcement becomes "not in the future" — a null reaching the validator afterward indicates a bug in the imputation step itself, not bad source data. |
| **CHK-4** | `Source_name` is now checked against a real enum (`CANONICAL_SOURCE_NAMES` in `00_config`) instead of just "not null". Built from the actual 9 sources in this codebase, not the input-checks sheet's stale 8-source list. |
| **CHK-5** | Missing `Summary` is *flagged*, not rejected, for any source expected to natively provide one (i.e. not in `SOURCES_WITHOUT_NATIVE_SUMMARY`). Written to a new `Dq_flags` column on the validated row — the article stays in `silver.articles`. |
| **CHK-7** | `Ingested_at` earlier than `Published_date` (a clock-skew/pipeline-bug signal) is also flagged into `Dq_flags`, plus logged to a small audit CSV in ADLS for manual review — the row is still kept, per the sheet's own "flagged, not dropped" instruction. |
| **CHK-9** | A new dedup pass, scoped to `dev.to` rows only, drops duplicate `Summary` text (keeping the row with the latest `Published_date`) — CHK-8/10/11 needed no new code since the existing `canonical_url` dedup already subsumes them. |
| **CDF** | `delta.enableChangeDataFeed` is now set on `silver.articles` at creation, and retrofitted onto an already-existing table if it's missing — so Gold can consume this table's change feed once that layer is built. |

**Deliberately NOT done here:** TR-11 through TR-14 (`trend_weight`, 
`Engagement`, `momentum_multiplier`, `final_trend_score`) are trend-scoring 
aggregations that belong in the Gold layer, not Silver harmonization — 
consistent with Gold still being a work in progress.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

In [0]:
%run ./00_config

# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string, 
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads 
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a 
storage account connection string — not Auto Loader over an `abfss://` 
path, and not a Unity Catalog External Location. That's a deliberate 
choice, not a step backward: Databricks Serverless blocks 
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`), 
which is what Auto Loader over `abfss://` would need without an External 
Location, and the External Location route needs grants (`READ FILES`, 
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you 
on a shared/class storage account. The Blob SDK never touches Spark's 
filesystem config at all -- it's a plain HTTPS call, same mechanism the 
live-fetch version (`py_work/`) already uses successfully against public 
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed 
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`, 
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup` 
creates the schemas (assumes the `fomoless` catalog itself already exists — 
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto 
Loader checkpoint tracking "what's already been read", incrementality 
comes from checking each landed file's blob name against what's already in 
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a 
re-run only pulls genuinely new files. **Silver stays incremental** via its 
own Structured Streaming checkpoint, stored in a small Unity Catalog 
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed 
storage under the catalog you already own, so it needs no extra grants 
either. That Volume also holds nothing else; dedup-merge audit logs go 
straight to ADLS via the same Blob SDK connection as bronze.

---
**SECURITY FIX (this revision):** a previous version of this notebook had 
a live Azure Storage account key hardcoded as the `connection_string` 
widget's default value. That key has been removed from this file — **if 
you're seeing this note, rotate that key in Azure Portal (Storage Account 
→ Access keys → Rotate) if you haven't already**, since it was exposed in 
a shared notebook. Going forward, set the connection string via the widget 
at run time or the `AZURE_STORAGE_CONNECTION_STRING` environment variable 
— never as a hardcoded default.

**New in this revision:** `CANONICAL_SOURCE_NAMES` (CHK-4 enum gate), 
`SOURCES_WITHOUT_NATIVE_SUMMARY` (CHK-5 conditional-completeness exception 
list), and `CLOCK_SKEW_LOG_PREFIX` (CHK-7 audit trail) — all consumed by 
`03_silver_transform`.

---
**Gold/Mongo revision:** added `ensure_cdf_enabled` (moved here from 
`03_silver_transform` since `04_gold_aggregation` now needs the identical 
logic — shared, not duplicated), `TREND_DECAY_HALF_LIFE_DAYS` and 
`MOMENTUM_MULTIPLIERS` (TR-11/TR-13), new `gold`/`mongo_sink` checkpoint 
paths, and `get_mongo_uri()` (mirrors `get_widget_or_env` — never 
hardcoded).

**⚠️ FLAGGED GAP — TR-13's momentum multiplier table is incomplete.** The 
source Transformation Rules sheet is cut off after the `BBC`/`The Guardian` 
→ `3.0` branch; the multiplier values for dev.to, Medium, Towards Data 
Science, Hacker News, Stack Overflow Blog, Stack Exchange, and KDnuggets 
were never visible in the source image. `MOMENTUM_MULTIPLIERS` below only 
has the one confirmed entry — every other source falls back to 
`DEFAULT_MOMENTUM_MULTIPLIER = 1.0` as an explicit placeholder. **This dict 
needs real values from the business before `04_gold_aggregation`'s trend 
scores are correct**, not just structurally complete.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection 
string (key 1 or key 2, either works). Settable via a Databricks widget 
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment 
variable -- never hardcode it directly in a cell.

## 3. Sources & schema

## 4. Catalog, schemas, tables & volume

## 5. Gold trend-scoring constants (TR-11, TR-13)

## 6. MongoDB connection (for `06_load_mongodb`)

Same pattern as the storage connection string above: widget first, then 
environment variable, never a hardcoded default. `MONGO_URI` should 
already contain the target database name (e.g. 
`.../fomoless?retryWrites=true&w=majority`); `MONGO_COLLECTION_NAME` names 
the collection within it.

## 7. Helpers

[2026-09-25 20:17:53 UTC] Config loaded — container: fomoless, catalog: fomoless


In [0]:
import json
import html
import re
import hashlib
from datetime import datetime
from urllib.parse import urlparse, urlunparse

import feedparser
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from azure.storage.blob import BlobServiceClient

## 1b. Schema evolution for the new `Dq_flags` column

**Corrected in this pass:** `Dq_flags` (CHK-5 / CHK-7) is new in this
revision and isn't part of the original `silver.articles` schema, so the
MERGE below needs Delta's automatic schema evolution turned on -- otherwise
a MERGE against an existing table predating this column fails with a
schema-mismatch error. The session-config approach
(`spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")`)
that used to live in the next cell **does not work on Databricks
Serverless** (confirmed by a real error -- see that cell). The fix is
`MERGE WITH SCHEMA EVOLUTION INTO`, Delta's serverless-compatible SQL
syntax for enabling schema evolution per-statement instead of per-session
-- used directly in `process_silver_batch`'s MERGE further down, so there's
nothing to set up front here anymore.

In [0]:
# BUG FIX (confirmed by a real CONFIG_NOT_AVAILABLE.SERVERLESS_DELTA_SCHEMA_AUTO_MERGE_ENABLED
# error): this spark.conf.set call does not work on Databricks Serverless at
# all -- Databricks' own KB confirms schema-evolution-related Spark configs
# aren't supported on serverless compute. Removed. The MERGE below now uses
# `MERGE WITH SCHEMA EVOLUTION INTO` (serverless-compatible SQL syntax)
# instead of relying on this session config, so nothing needs to run here.

In [0]:
# Captured once per notebook run (not per micro-batch) so every row this run
# produces -- across however many micro-batches `availableNow` ends up
# splitting the backlog into -- shares the same Cleaned_at.
CLEANED_AT = pd.Timestamp.now(tz=RIYADH_TZ)

## 2. Harmonization logic

In [0]:
def synthetic_id(key: str) -> int:
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) % 2_000_000_000 + 1


def normalize_url(url):
    if is_missing(url):
        return None
    u = str(url).strip().lower()
    if u.endswith("/"):
        u = u[:-1]
    return u


def fix_url_scheme(url):
    """CHK-2 ("fix automatically"): upgrade a plain http:// URL to https://
    rather than rejecting it outright. Anything without a recognizable
    http(s) scheme is left untouched -- validate_url still rejects those,
    since there's no safe automatic fix for a URL with no scheme at all."""
    if is_missing(url):
        return url
    parsed = urlparse(str(url).strip())
    if parsed.scheme.lower() == "http":
        return urlunparse(parsed._replace(scheme="https"))
    return url


def clean_url(url):
    if is_missing(url):
        return None
    fixed = fix_url_scheme(url)
    return str(fixed).strip()


def clean_html_text(text):
    if is_missing(text):
        return None
    no_tags = re.sub(r"<[^>]+>", " ", str(text))
    decoded = html.unescape(no_tags)
    return re.sub(r"\s+", " ", decoded).strip()


def clean_plain_text(text):
    if is_missing(text):
        return None
    return re.sub(r"\s+", " ", str(text)).strip()


def canonical_url_fn(url, title):
    norm = normalize_url(url)
    if norm:
        return norm
    if is_missing(title):
        return None
    return "title:" + re.sub(r"\s+", " ", str(title).strip().lower())


def parse_date_iso(value):
    if is_missing(value):
        return None
    try:
        return pd.to_datetime(value, utc=True)
    except Exception:
        return None


def parse_date_rfc822(value):
    if is_missing(value):
        return None
    try:
        return pd.to_datetime(value, utc=True)
    except Exception:
        return None


def parse_date_epoch(value):
    if is_missing(value):
        return None
    try:
        return pd.to_datetime(int(value), unit="s", utc=True)
    except Exception:
        return None


def impute_published_date(value):
    """TR-8: a null Published_date is imputed with 'today' -- using this
    run's CLEANED_AT (Riyadh-local) rather than server UTC 'now', to stay
    consistent with the rest of the pipeline's timezone convention. This
    directly resolves the CHK-3 vs TR-8 conflict noted at the top of this
    notebook: by the time validate_published_date runs, a null value here
    would mean this imputation step itself broke, not that the source data
    was incomplete."""
    if is_missing(value):
        return CLEANED_AT
    return value


def pad_tags(tag_list):
    tags = [t for t in (tag_list or []) if not is_missing(t)][:6]
    tags += [None] * (6 - len(tags))
    return tags


def to_riyadh_tz(ts):
    """Converts a timestamp to RIYADH_TZ, tolerating both tz-aware and
    tz-naive input. bronze's `ingested_at` is always written as a genuinely
    UTC-aware value (see 02_bronze_ingestion.py, `_metadata.file_modification_time`),
    but on Databricks Serverless (Spark Connect) round-tripping it through a
    Delta table and back via .toPandas() can strip the tzinfo, leaving a
    naive value that still represents a UTC instant -- .astimezone() alone
    raises on that ("Cannot convert tz-naive Timestamp, use tz_localize to
    localize"), so a naive value is localized as UTC first rather than
    assumed already local."""
    if ts is None:
        return None
    if getattr(ts, "tzinfo", None) is None:
        ts = pd.Timestamp(ts).tz_localize("UTC")
    return ts.astimezone(RIYADH_TZ)


def make_row(article_id, title, source_name, author, published_date, url,
             tags, summary, ingested_at, html_summary=False,
             raw_upvotes=None, raw_comments=None):
    """raw_upvotes/raw_comments are new in this revision -- added so
    04_gold_aggregation's TR-12 has real numbers to consolidate. Deliberately
    ONE generic pair rather than six per-source-named fields (reactions_count,
    points, score, etc.) since TR-12 only ever needs their sum, never the
    per-source distinction. Only dev.to, Hacker News, and Stack Exchange pass
    non-None values; every other source stays None, which is exactly what
    TR-12's COALESCE(..., 0) / editorial-NULL logic expects."""
    padded = pad_tags(tags)
    has_tag = any(t is not None for t in padded)
    clean_summary = clean_html_text(summary) if html_summary else clean_plain_text(summary)
    return {
        "Article_ID": article_id,
        "Title": clean_plain_text(title),
        "Source_name": source_name,
        "Author": clean_plain_text(author) if author else None,
        "Published_date": impute_published_date(published_date),
        "Url": clean_url(url),
        "Tag_1": padded[0], "Tag_2": padded[1], "Tag_3": padded[2],
        "Tag_4": padded[3], "Tag_5": padded[4], "Tag_6": padded[5],
        "Has_tag": has_tag,
        "Summary": clean_summary,
        "Ingested_at": to_riyadh_tz(ingested_at),
        "Cleaned_at": CLEANED_AT,
        "canonical_url": canonical_url_fn(url, title),
        "raw_upvotes": raw_upvotes,
        "raw_comments": raw_comments,
    }

## 3. Per-source mappers

The new `raw_upvotes`/`raw_comments` fields (dev.to, Hacker News, Stack 
Exchange only) live in the three JSON mappers below. RSS mappers 
(`map_medium_like`, `map_bbc`, `map_so_blog`, `map_kdnuggets`) are 
untouched — they safely default to `None` via `make_row`'s kwargs.

In [0]:
def map_devto(item, ingested_at):
    return make_row(
        article_id=item["id"], title=item["title"], source_name="dev.to",
        author=item.get("user", {}).get("name"),
        published_date=parse_date_iso(item.get("published_at")),
        url=item.get("url"), tags=item.get("tag_list"), summary=item.get("description"),
        ingested_at=ingested_at,
        # TR-12: dev.to's own field names, confirmed against its public API.
        raw_upvotes=item.get("public_reactions_count"),
        raw_comments=item.get("comments_count"),
    )


def map_medium_like(entry, source_name, ingested_at, html_summary=True):
    return make_row(
        article_id=synthetic_id(entry.get("link") or entry.get("title", "")),
        title=entry.get("title"), source_name=source_name, author=entry.get("author"),
        published_date=parse_date_rfc822(entry.get("published")),
        url=entry.get("link"),
        tags=[t.get("term") for t in entry.get("tags", [])] if entry.get("tags") else [],
        summary=entry.get("summary"), ingested_at=ingested_at, html_summary=html_summary,
    )


def map_guardian(item, ingested_at):
    fields = item.get("fields", {})
    return make_row(
        article_id=synthetic_id(item.get("id", "")),
        title=item.get("webTitle"), source_name="The Guardian",
        author=fields.get("byline"),
        published_date=parse_date_iso(item.get("webPublicationDate")),
        url=item.get("webUrl"),
        tags=[item.get("sectionName")] if item.get("sectionName") else [],
        summary=fields.get("trailText"), ingested_at=ingested_at, html_summary=True,
    )


def map_bbc(entry, ingested_at):
    return make_row(
        article_id=synthetic_id(entry.get("link") or entry.get("title", "")),
        title=entry.get("title"), source_name="BBC", author=None,
        published_date=parse_date_rfc822(entry.get("published")),
        url=entry.get("link"), tags=[], summary=entry.get("summary"),
        ingested_at=ingested_at,
    )


def map_hackernews(item, ingested_at):
    return make_row(
        article_id=int(item["objectID"]), title=item.get("title"), source_name="Hacker News",
        author=item.get("author"),
        published_date=parse_date_iso(item.get("created_at")),
        url=item.get("url"), tags=[], summary=None,
        ingested_at=ingested_at,
        # TR-12: Algolia HN Search API's own field names.
        raw_upvotes=item.get("points"),
        raw_comments=item.get("num_comments"),
    )


def map_so_blog(entry, ingested_at):
    return make_row(
        article_id=synthetic_id(entry.get("link") or entry.get("title", "")),
        title=entry.get("title"), source_name="Stack Overflow Blog", author=entry.get("author"),
        published_date=parse_date_rfc822(entry.get("published")),
        url=entry.get("link"), tags=[], summary=entry.get("summary"),
        ingested_at=ingested_at,
    )


def map_stackexchange(item, ingested_at):
    return make_row(
        article_id=item["question_id"], title=item.get("title"), source_name="Stack Exchange",
        author=item.get("owner", {}).get("display_name"),
        published_date=parse_date_epoch(item.get("creation_date")),
        url=item.get("link"), tags=item.get("tags"), summary=None,
        ingested_at=ingested_at,
        # TR-12: Stack Exchange API's own field names.
        raw_upvotes=item.get("score"),
        raw_comments=item.get("answer_count"),
    )


def map_kdnuggets(entry, ingested_at):
    return make_row(
        article_id=synthetic_id(entry.get("link") or entry.get("title", "")),
        title=entry.get("title"), source_name="KDnuggets", author=entry.get("author"),
        published_date=parse_date_rfc822(entry.get("published")),
        url=entry.get("link"),
        tags=[t.get("term") for t in entry.get("tags", [])] if entry.get("tags") else [],
        summary=entry.get("summary"), ingested_at=ingested_at,
    )


JSON_MAPPERS = {
    "devto": lambda payload, ingested_at: [map_devto(item, ingested_at) for item in payload],
    "hackernews": lambda payload, ingested_at: [map_hackernews(item, ingested_at) for item in payload.get("hits", [])],
    "stackexchange": lambda payload, ingested_at: [map_stackexchange(item, ingested_at) for item in payload.get("items", [])],
    "guardian": lambda payload, ingested_at: [map_guardian(item, ingested_at) for item in payload.get("response", {}).get("results", [])],
}

RSS_MAPPERS = {
    "medium": lambda feed, ingested_at: [map_medium_like(e, "Medium", ingested_at) for e in feed.entries],
    "towardsdatascience": lambda feed, ingested_at: [map_medium_like(e, "Towards Data Science", ingested_at) for e in feed.entries],
    "bbc": lambda feed, ingested_at: [map_bbc(e, ingested_at) for e in feed.entries],
    "stackoverflow_blog": lambda feed, ingested_at: [map_so_blog(e, ingested_at) for e in feed.entries],
    "kdnuggets": lambda feed, ingested_at: [map_kdnuggets(e, ingested_at) for e in feed.entries],
}

## 4. Validation

`validate_row` is the single gate every parsed row passes through before 
`df_checked` is built. New in this revision: `validate_source_name` 
(CHK-4, replacing a bare not-null check) and an updated 
`validate_published_date` (CHK-3, now meaningful now that TR-8 imputation 
guarantees the value is non-null by the time this runs). 
`compute_dq_flags` (CHK-5, CHK-7) runs *after* the pass/fail split, only 
against rows that already passed — it never causes a rejection.

In [0]:
def clean_medium_summary(text, source_name):
    if pd.isna(text):
        return text
    if str(source_name).strip() not in MEDIUM_SOURCES:
        return text
    no_tags = re.sub(r"<[^>]+>", " ", str(text))
    decoded = html.unescape(no_tags)
    return re.sub(r"\s+", " ", decoded).strip()


def validate_article_id(value, seen_ids: set) -> list:
    reasons = []
    if is_missing(value):
        reasons.append("Article_ID is null (required)")
        return reasons
    try:
        int_value = int(float(value))
    except (ValueError, TypeError):
        reasons.append("Article_ID is not a valid integer")
        return reasons
    if int_value <= 0:
        reasons.append("Article_ID must be a positive integer")
    if int_value in seen_ids:
        reasons.append(f"Article_ID {int_value} is a duplicate")
    else:
        seen_ids.add(int_value)
    return reasons


def validate_required_string(value, column_name: str) -> list:
    return [f"{column_name} is null (required)"] if is_missing(value) else []


def validate_source_name(value) -> list:
    """CHK-4: Source_name must be one of the pipeline's real canonical
    names, not just non-null. A failure here means a mapper is assigning
    the wrong literal -- i.e. a pipeline bug, not a data-quality issue with
    the source itself (hence the sheet's '0% acceptable' tolerance)."""
    reasons = validate_required_string(value, "Source_name")
    if reasons:
        return reasons
    if str(value).strip() not in CANONICAL_SOURCE_NAMES:
        reasons.append(f"Source_name '{value}' is not a recognized source (pipeline bug)")
    return reasons


def validate_published_date(value) -> list:
    """CHK-3: by the time this runs, TR-8 imputation (see
    impute_published_date) should have already replaced any null with
    CLEANED_AT, so a null reaching here means that imputation step itself
    broke -- flagged distinctly from a genuinely-invalid or future date."""
    if is_missing(value):
        return ["Published_date is null even after TR-8 imputation (pipeline bug)"]
    try:
        ts = pd.to_datetime(value)
    except (ValueError, TypeError):
        return ["Published_date is not a valid datetime"]
    ts_utc = ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")
    if ts_utc > pd.Timestamp.now(tz="UTC"):
        return ["Published_date is in the future"]
    return []


def validate_timestamp_required(value, column_name: str) -> list:
    if is_missing(value):
        return [f"{column_name} is null (required)"]
    try:
        pd.to_datetime(value)
    except (ValueError, TypeError):
        return [f"{column_name} is not a valid datetime"]
    return []


def validate_url(value) -> list:
    """CHK-2: by this point clean_url has already upgraded a plain http://
    URL to https:// (see fix_url_scheme), so a URL still failing this check
    was either missing entirely or had no recognizable http(s) scheme at
    all -- there's no safe automatic fix for that case."""
    if is_missing(value):
        return ["Url is null (required)"]
    parsed = urlparse(str(value))
    if parsed.scheme.lower() != "https":
        return ["Url must start with https"]
    return []


def validate_has_tag(value) -> list:
    if is_missing(value):
        return ["Has_tag is null (required)"]
    normalized = str(value).strip().lower()
    if normalized not in TRUE_VALUES | FALSE_VALUES:
        return ["Has_tag is not a valid boolean (expected TRUE/FALSE)"]
    return []


def normalize_has_tag(value) -> bool:
    return str(value).strip().lower() in TRUE_VALUES


def validate_row(row: pd.Series, seen_ids: set) -> list:
    reasons = []
    reasons += validate_article_id(row.get("Article_ID"), seen_ids)
    reasons += validate_required_string(row.get("Title"), "Title")
    reasons += validate_source_name(row.get("Source_name"))
    reasons += validate_published_date(row.get("Published_date"))
    reasons += validate_url(row.get("Url"))
    reasons += validate_has_tag(row.get("Has_tag"))
    reasons += validate_timestamp_required(row.get("Ingested_at"), "Ingested_at")
    reasons += validate_timestamp_required(row.get("Cleaned_at"), "Cleaned_at")
    return reasons


def compute_dq_flags(row) -> str:
    """CHK-5 + CHK-7: soft flags on rows that ALREADY PASSED validate_row.
    These never cause a rejection -- the row stays in silver.articles,
    flagged for manual review, per both checks' own 'flagged, not dropped'
    instruction (which otherwise conflicts with routing every failing row
    to silver_rejected -- see the note at the top of this notebook)."""
    flags = []

    source = str(row.get("Source_name") or "").strip()
    if is_missing(row.get("Summary")) and source not in SOURCES_WITHOUT_NATIVE_SUMMARY:
        flags.append("CHK-5: summary missing for a source that normally provides one")

    ingested = row.get("Ingested_at")
    published = row.get("Published_date")
    if not is_missing(ingested) and not is_missing(published):
        try:
            if pd.to_datetime(ingested) < pd.to_datetime(published):
                flags.append("CHK-7: Ingested_at is before Published_date (clock skew)")
        except (ValueError, TypeError):
            pass

    return "; ".join(flags) if flags else None

## 5. Per-micro-batch processing

Runs once per new chunk of bronze rows. `Article_ID` duplicate-checking 
(`validate_article_id`'s `seen_ids`) is intentionally scoped to *this 
batch only* — a real article reappearing in a later, separate run isn't a 
duplicate to reject, it's an update, and that's what the `MERGE` below is 
for.

**New in this revision:** a CHK-9 dedup pass (dev.to only, keyed on 
`Summary`) runs right after the existing CHK-6 canonical-URL dedup; 
`Dq_flags` is computed for validated rows only, with CHK-7 flags also 
logged to a small audit CSV; and Delta Change Data Feed is ensured on 
`silver.articles` via the now-shared `ensure_cdf_enabled` (moved to 
`00_config` this revision, since `04_gold_aggregation` needs the identical 
logic for `gold.articles`).

In [0]:
def dedupe_devto_by_summary(df):
    """CHK-9: within dev.to rows only, two articles sharing identical
    Summary text are treated as duplicates -- keep the one with the latest
    Published_date, silently drop the rest (the sheet marks this
    'not logged', unlike CHK-6's merge log)."""
    is_devto = df["Source_name"] == "dev.to"
    df_devto = df[is_devto].copy()
    df_rest = df[~is_devto].copy()

    has_summary = df_devto["Summary"].notna()
    df_devto_keyed = df_devto[has_summary].sort_values("Published_date", ascending=False, na_position="last")
    df_devto_deduped = df_devto_keyed.drop_duplicates(subset="Summary", keep="first")
    df_devto_unkeyed = df_devto[~has_summary]

    return pd.concat([df_devto_deduped, df_devto_unkeyed, df_rest], ignore_index=True)


def process_silver_batch(batch_df, batch_id: int) -> None:
    df_latest = batch_df.toPandas()
    if df_latest.empty:
        log(f"[silver batch {batch_id}] No new bronze rows — nothing to do.")
        return
    log(f"[silver batch {batch_id}] Processing {len(df_latest)} new bronze row(s)")

    all_rows = []
    for _, bronze_row in df_latest.iterrows():
        folder = bronze_row["source"]
        kind = bronze_row["kind"]
        ingested_at = bronze_row["ingested_at"]
        try:
            if kind == "json":
                payload = json.loads(bronze_row["raw_content"])
                rows = JSON_MAPPERS[folder](payload, ingested_at)
            else:
                feed = feedparser.parse(bronze_row["raw_content"])
                rows = RSS_MAPPERS[folder](feed, ingested_at)
            all_rows.extend(rows)
            log(f"[silver batch {batch_id}] {folder}: {len(rows)} rows parsed")
        except Exception as exc:
            log(f"[silver batch {batch_id}] ⚠️  {folder} failed to parse: {exc}")

    df_all = pd.DataFrame(all_rows, columns=TARGET_COLUMNS)
    if df_all.empty:
        log(f"[silver batch {batch_id}] No articles parsed from this batch.")
        return

    # CHK-6 -- dedupe within this batch on canonical_url.
    has_key = df_all["canonical_url"].notna()
    df_keyed = df_all[has_key].copy()
    df_unkeyed = df_all[~has_key].copy()
    df_keyed = df_keyed.sort_values("Published_date", ascending=False, na_position="last")
    df_deduped = df_keyed.drop_duplicates(subset="canonical_url", keep="first")
    dropped = df_keyed.drop(df_deduped.index)
    df_cleaned = pd.concat([df_deduped, df_unkeyed], ignore_index=True)

    log(f"[silver batch {batch_id}] Kept {len(df_cleaned)} row(s), merged {len(dropped)} in-batch duplicate(s) (CHK-6)")
    if len(dropped) > 0:
        # Uploaded straight to ADLS with the Blob SDK -- same connection
        # string as bronze, plain HTTPS, no Unity Catalog grant needed.
        merge_log_container = BlobServiceClient.from_connection_string(CONNECTION_STRING).get_container_client(CONTAINER_NAME)
        merge_log_path = f'{MERGE_LOG_PREFIX}/dedup_merges_batch{batch_id}_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.csv'
        merge_log_bytes = dropped[["Article_ID", "Title", "Source_name", "canonical_url"]].to_csv(index=False).encode("utf-8")
        merge_log_container.upload_blob(name=merge_log_path, data=merge_log_bytes, overwrite=True)
        log(f"[silver batch {batch_id}] Merge log written to {merge_log_path}")

    # CHK-9 -- dev.to-only dedup on Summary text (not logged, per the sheet).
    before_ct = len(df_cleaned)
    df_cleaned = dedupe_devto_by_summary(df_cleaned)
    devto_dropped = before_ct - len(df_cleaned)
    if devto_dropped > 0:
        log(f"[silver batch {batch_id}] Dropped {devto_dropped} dev.to row(s) with duplicate Summary (CHK-9)")

    # Medium/TDS summary cleanup.
    df_cleaned["Summary"] = df_cleaned.apply(
        lambda r: clean_medium_summary(r.get("Summary"), r.get("Source_name")), axis=1
    )

    # Validation -- seen_ids starts empty every batch, on purpose; see the
    # markdown cell above.
    seen_article_ids = set()
    rejection_reasons = [validate_row(row, seen_article_ids) for _, row in df_cleaned.iterrows()]

    df_checked = df_cleaned.copy()
    df_checked["_rejection_reasons"] = rejection_reasons
    df_checked["_is_valid"] = df_checked["_rejection_reasons"].apply(len).eq(0)

    # canonical_url is KEPT here -- it's silver's MERGE key, so it has to be
    # persisted for the next batch to match against.
    df_validated = df_checked[df_checked["_is_valid"]].drop(columns=["_rejection_reasons", "_is_valid"]).copy()
    if "Has_tag" in df_validated.columns:
        df_validated["Has_tag"] = df_validated["Has_tag"].apply(normalize_has_tag)
    if "Article_ID" in df_validated.columns:
        df_validated["Article_ID"] = df_validated["Article_ID"].astype("int64")

    # CHK-5 / CHK-7 -- soft flags, computed only on rows that already
    # passed validate_row. Never causes a rejection.
    if len(df_validated) > 0:
        df_validated["Dq_flags"] = df_validated.apply(compute_dq_flags, axis=1)

        clock_skew_rows = df_validated[df_validated["Dq_flags"].fillna("").str.contains("CHK-7")]
        if len(clock_skew_rows) > 0:
            skew_container = BlobServiceClient.from_connection_string(CONNECTION_STRING).get_container_client(CONTAINER_NAME)
            skew_log_path = f'{CLOCK_SKEW_LOG_PREFIX}/clock_skew_batch{batch_id}_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.csv'
            skew_log_bytes = clock_skew_rows[["Article_ID", "Title", "Source_name", "Ingested_at", "Published_date"]].to_csv(index=False).encode("utf-8")
            skew_container.upload_blob(name=skew_log_path, data=skew_log_bytes, overwrite=True)
            log(f"[silver batch {batch_id}] {len(clock_skew_rows)} row(s) flagged for CHK-7 clock skew — logged to {skew_log_path}")

    df_rejected = df_checked[~df_checked["_is_valid"]].drop(columns=["_is_valid", "canonical_url"]).copy()
    df_rejected["rejection_reason"] = df_rejected["_rejection_reasons"].apply(lambda r: "; ".join(r))
    df_rejected = df_rejected.drop(columns=["_rejection_reasons"])

    log(f"[silver batch {batch_id}] validated={len(df_validated)} rejected={len(df_rejected)}")

    if len(df_validated) > 0:
        spark_validated = spark.createDataFrame(df_validated)
        if spark.catalog.tableExists(TABLES["silver"]):
            ensure_cdf_enabled(TABLES["silver"])
            # BUG FIX: DeltaTable.merge()'s whenMatchedUpdateAll() needs
            # schema-evolution support that isn't available on serverless
            # (see the cell above) -- MERGE WITH SCHEMA EVOLUTION INTO is
            # the serverless-compatible SQL equivalent. spark.sql() needs a
            # named relation, not a DataFrame object, hence the temp view
            # (batch-scoped name + dropped right after, so concurrent
            # micro-batches -- if this ever runs somewhere that allows
            # them -- can't collide on the same view name).
            view_name = f"silver_updates_batch_{batch_id}"
            spark_validated.createOrReplaceTempView(view_name)
            spark.sql(f"""
                MERGE WITH SCHEMA EVOLUTION INTO {TABLES['silver']} AS silver
                USING {view_name} AS updates
                ON silver.canonical_url = updates.canonical_url
                WHEN MATCHED THEN UPDATE SET *
                WHEN NOT MATCHED THEN INSERT *
            """)
            spark.catalog.dropTempView(view_name)
        else:
            (
                spark_validated.write.format("delta")
                .option("delta.enableChangeDataFeed", "true")
                .saveAsTable(TABLES["silver"])
            )
        log(f"[silver batch {batch_id}] Merged {len(df_validated)} row(s) into silver")

    if len(df_rejected) > 0:
        (
            spark.createDataFrame(df_rejected).write.format("delta")
            .mode("append").option("mergeSchema", "true").saveAsTable(TABLES["silver_rejected"])
        )
        log(f"[silver batch {batch_id}] Appended {len(df_rejected)} row(s) to silver_rejected")

## 6. Stream new bronze rows through it

In [0]:
query = (
    spark.readStream.table(TABLES["bronze"])
    .writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", CHECKPOINTS["silver"])
    .trigger(availableNow=True)
    .start()
)

run_stream(query, "silver")

[2026-09-25 20:19:13 UTC] Stream 'silver' finished.


In [0]:
log(f"{TABLES['silver']} total rows: {row_count(TABLES['silver'])}")

# silver_rejected is only created the first time a row is actually rejected
# -- on a run where nothing new was processed (or everything that was
# processed happened to validate cleanly), it may not exist yet.
if spark.catalog.tableExists(TABLES["silver_rejected"]):
    display(
        spark.table(TABLES["silver_rejected"])
        .groupBy("rejection_reason")
        .count()
        .orderBy(F.desc("count"))
    )
else:
    log(f"{TABLES['silver_rejected']} doesn't exist yet — no rows have been rejected so far.")

[2026-09-25 20:19:15 UTC] fomoless.silver.articles total rows: 2731


rejection_reason,count
Url is null (required),540


## 7. Dq_flags summary (new in this revision)

A quick view of how many validated rows carry a CHK-5 and/or CHK-7 flag — 
useful for deciding whether either threshold needs tuning before Gold 
consumes this table.

In [0]:
if spark.catalog.tableExists(TABLES["silver"]) and "Dq_flags" in spark.table(TABLES["silver"]).columns:
    display(
        spark.table(TABLES["silver"])
        .withColumn("has_flag", F.col("Dq_flags").isNotNull())
        .groupBy("has_flag")
        .count()
    )
else:
    log("Dq_flags column not present yet on silver.articles (no batch has produced a flagged row).")

has_flag,count
false,2729
true,2
